In [ ]:
! pip install pytorch-forecasting pytorch-lightning

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.tuner import Tuner
import matplotlib.pyplot as plt

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, Baseline
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.data.encoders import MultiNormalizer, TorchNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE, SMAPE
from sklearn.model_selection import train_test_split

# Constants
context_length = 512
forecast_length = 96
target_columns = [
    'COOLANT_TEMPERATURE ()',
    'ENGINE_RPM ()',
    'VEHICLE_SPEED ()',
    'THROTTLE ()',
    'ENGINE_LOAD ()',
    'INTAKE_MANIFOLD_PRESSURE ()',
]
time_col = 'ENGINE_RUN_TINE ()'
path = "/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/data/carOBD/obdiidata"

In [ ]:
df_list = []
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(f'{path}/{file}', index_col=False)
        df['drive_id'] = file
        df_list.append(df)

print(f'{len(df_list)} files loaded out of {len([f for f in os.listdir(path) if f.endswith(".csv")])}')

In [ ]:
def remove_zero_variance_columns(df: pd.DataFrame, exclude_cols: list[str] = None) -> pd.DataFrame:
    """
    Compute std of each std-computable column (numeric only)
    """
    if exclude_cols is None:
        exclude_cols = []
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    cols_to_check = [col for col in numeric_cols if col not in exclude_cols]
  
    std_df = df[cols_to_check].std()
    zero_variance_cols = std_df[std_df == 0].index.tolist()
  
    print(f'{len(zero_variance_cols)} columns with zero variance: {zero_variance_cols}')
  
    if len(zero_variance_cols) > 0:
        df = df.drop(columns=zero_variance_cols)
  
    return df

In [ ]:
def mean_fill_missing_timestamps_and_remove_duplicates(df: pd.DataFrame, time_col: str, id_cols: list[str] = None) -> pd.DataFrame:
    """
    Remove duplicate timestamps by averaging all numeric columns for each unique timestamp.
    This preserves the overall statistics while removing duplicate entries.
    
    Note: The time column itself is not averaged (it becomes the group key).
    Only numeric columns are averaged when multiple rows share the same timestamp.
    """
    if id_cols is None:
        id_cols = []
    
    existing_id_cols = [col for col in id_cols if col in df.columns]
    
    group_cols = [time_col] + existing_id_cols
    
    agg_dict = {}
    for col in df.columns:
        if col not in group_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                agg_dict[col] = 'mean'
            else:
                agg_dict[col] = 'first'
  
    df_clean = df.groupby(group_cols, as_index=False).agg(agg_dict)
  
    return df_clean

In [ ]:
def downsample(df, time_col, source_file_col, downsample_factor=2):
    result_dfs = []
    
    for source_file in df[source_file_col].unique():
        file_df = df[df[source_file_col] == source_file].copy()
        
        if len(file_df) < downsample_factor * 2:
            continue
        
        file_df = file_df.sort_values(time_col).reset_index(drop=True)
        
        # Simple decimation without pre-smoothing
        downsampled = file_df.iloc[::downsample_factor].copy()
        downsampled[time_col] = np.arange(len(downsampled)) * downsample_factor
        
        result_dfs.append(downsampled.reset_index(drop=True))
    
    return pd.concat(result_dfs, ignore_index=True)

In [ ]:
def filter_long_drives(df, id_col='drive_id', min_length=608):
    """Keep only drives long enough for your context window"""
    drive_lengths = df.groupby(id_col).size()
    valid_drives = drive_lengths[drive_lengths >= min_length].index
    
    print(f"Keeping {len(valid_drives)}/{df[id_col].nunique()} drives")
    print(f"Dropped {len(df) - df[df[id_col].isin(valid_drives)].shape[0]} timesteps")
    
    return df[df[id_col].isin(valid_drives)].reset_index(drop=True)

In [ ]:
# Combine all dataframes
data = pd.concat(df_list, ignore_index=True)

# Clean up
print(f"Total samples: {len(data):,}")
print(f"Unique drives: {data['drive_id'].nunique()}")

data = mean_fill_missing_timestamps_and_remove_duplicates(data, time_col=time_col, id_cols=["drive_id"])
data = remove_zero_variance_columns(data, exclude_cols=["drive_id"])
data = downsample(
    data,
    time_col=time_col,
    source_file_col='drive_id',
    downsample_factor=1
)

data = filter_long_drives(data, min_length=context_length + forecast_length)

# Add derivative features
data['speed_change'] = data.groupby('drive_id')['VEHICLE_SPEED ()'].diff()
data['rpm_change'] = data.groupby('drive_id')['ENGINE_RPM ()'].diff()

In [ ]:
def add_cross_channel_features(data, target_columns):
    """
    Engineer features that capture cross-channel relationships.
    Add these as conditional columns.
    """
    # RPM-to-Speed ratio (gear indicator)
    if 'ENGINE_RPM ()' in data.columns and 'VEHICLE_SPEED ()' in data.columns:
        data['RPM_SPEED_RATIO'] = data['ENGINE_RPM ()'] / (data['VEHICLE_SPEED ()'] + 1)
    
    # Throttle-to-Load ratio (efficiency indicator)
    if 'THROTTLE ()' in data.columns and 'ENGINE_LOAD ()' in data.columns:
        data['THROTTLE_LOAD_RATIO'] = data['THROTTLE ()'] / (data['ENGINE_LOAD ()'] + 1)
    
    # Speed-based categories
    if 'VEHICLE_SPEED ()' in data.columns:
        data['IS_IDLE'] = (data['VEHICLE_SPEED ()'] < 5).astype(float)
        data['IS_HIGHWAY'] = (data['VEHICLE_SPEED ()'] > 60).astype(float)
    
    # RPM acceleration
    if 'ENGINE_RPM ()' in data.columns:
        data['RPM_ACCEL'] = data.groupby('drive_id')['ENGINE_RPM ()'].diff().fillna(0)
    
    return data

# Apply cross-channel features before preprocessing
data = add_cross_channel_features(data, target_columns)
print("Added cross-channel features")

In [ ]:
# ---------------------------------------------------------------------
# 1. Ensure time index and sorting
# ---------------------------------------------------------------------

# Ensure data is sorted
data = data.sort_values(["drive_id", time_col]).reset_index(drop=True)

# Use existing time column as time_idx (must be integer)
data["time_idx"] = data[time_col].astype("int64")

# ---------------------------------------------------------------------
# 2. Train/validation/test split by time_idx
# ---------------------------------------------------------------------
max_time_idx = data["time_idx"].max()

train_cutoff = int(max_time_idx * 0.70)
val_cutoff   = int(max_time_idx * 0.85)

train_data = data[data["time_idx"] <= train_cutoff]
val_data   = data[(data["time_idx"] > train_cutoff) & (data["time_idx"] <= val_cutoff)]
test_data  = data[data["time_idx"] > val_cutoff]

print(
    f"Train samples: {len(train_data)}, "
    f"Val samples: {len(val_data)}, "
    f"Test samples: {len(test_data)}"
)

In [ ]:
# ---------------------------------------------------------------------
# 3. Build TimeSeriesDataSet
# ---------------------------------------------------------------------

# all real-valued columns except identifiers and targets
real_cols = [
    c for c in data.columns
    if c not in ["drive_id", "time_idx"] and c not in target_columns # identifiers and targets
]

# time_varying_unknown_reals are things we don't know in the future (targets + other features)
time_varying_unknown_reals = target_columns + real_cols

# time_varying_known_reals are things we know (time_idx)
time_varying_known_reals = ["time_idx"]

# Explicit multi-target normalizer
target_normalizer = MultiNormalizer(
    [TorchNormalizer()] * len(target_columns)
)

# 1. Training dataset
training_dataset = TimeSeriesDataSet(
    data=train_data,
    time_idx="time_idx",
    target=target_columns,                 # multi-target forecasting
    group_ids=["drive_id"],                # one series per drive
    max_encoder_length=context_length,
    max_prediction_length=forecast_length,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
    static_categoricals=None,             
    static_reals=None,
    target_normalizer=target_normalizer,
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# 2. Validation and test datasets
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data=val_data,
    stop_randomization=True,   # deterministic encoder/decoder lengths
)

test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data=test_data,
    stop_randomization=True,
)

# 3. Dataloaders
batch_size = 64

train_dataloader = training_dataset.to_dataloader(
    train=True,
    batch_size=batch_size,
    num_workers=0,
)

val_dataloader = validation_dataset.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=0,
)

test_dataloader = test_dataset.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=0,
)

In [ ]:
# ---------------------------------------------------------------------
# 4. Define and train TemporalFusionTransformer
# ---------------------------------------------------------------------

# Reproducibility
pl.seed_everything(42)

# Configure accelerator
accelerator = "auto"

# 1. Define model from dataset
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=1e-3,
    hidden_size=64,              # model capacity
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=32,
    loss=QuantileLoss(),         # multi-quantile loss for probabilistic forecasts
    log_interval=50,
    log_val_interval=1,
    reduce_on_plateau_patience=3,
)

print(f"Number of parameters in model: {tft.size()/1e3:.1f}k")

# 2. Trainer
trainer = pl.Trainer(
    max_epochs=30,
    accelerator=accelerator,
    gradient_clip_val=0.1,
    enable_checkpointing=True,
    callbacks=[
        EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
    ],
)

# 3. Train
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

# Load best checkpoint
best_tft = TemporalFusionTransformer.load_from_checkpoint(
    trainer.checkpoint_callback.best_model_path
)

In [ ]:
# ---------------------------------------------------------------------
# 5. Predict future sensor values
# ---------------------------------------------------------------------

# 1. Point or quantile forecasts on the test set
# By default, returns median prediction (quantile 0.5) for each target
test_predictions = best_tft.predict(
    test_dataloader,  # shape: (batch, max_prediction_length, n_targets)
)

print(f"Predictions shape: {test_predictions.shape}")

# 2. Get raw output and map back to time/drive for inspection
raw_predictions, x = best_tft.predict(
    test_dataloader,
    mode="raw",
    return_x=True,
)

# Example: take first batch and first sample in that batch
sample_idx = 0
decoder_time = x["decoder_time_idx"][sample_idx].cpu().numpy()
decoder_drive = training_dataset.x_to_index(x)["drive_id"].iloc[sample_idx]  # drive id for this sample

# predictions for that sample: (prediction_length, n_targets)
sample_pred = raw_predictions["prediction"][sample_idx].cpu().numpy()

print("Drive:", decoder_drive)
print("Decoder time indices:", decoder_time)
print("Prediction shape (quantiles included):", sample_pred.shape)

# Visualize one sample
idx = 0  # plot first sample
best_tft.plot_prediction(x, raw_predictions, idx=idx, add_loss_to_title=True)
plt.show()